# Pixtall generation lifecycle — function reference

This notebook explains the complete path of one generation request: API submission, validation, identity, pricing, credit reservation, queued processing, backend execution, settlement, and final inspection.

Each section documents the functions in the following code cell, including their inputs, outputs, database effects, and role in the system.

## Runtime setup and configuration functions

- `read_colab_secret(name)` reads a value from the environment or Colab Secrets and uses hidden keyboard input only as a fallback. Its input is a secret name; its output is the secret string. It never prints the value.
- `normalize_database_url(raw_url)` validates a PostgreSQL URL, selects the Psycopg 3 driver, and adds TLS when missing. Its input is the stored URL; its output is the SQLAlchemy-compatible URL.

Before execution, start `make api` and `npm run dev`, keep `make worker` stopped, and store the Supabase Session Pooler URL in Colab Secrets as `DATABASE_URL`. Click **Generate** in the frontend before continuing through the lifecycle cells.

In [ ]:
%pip install -q "sqlalchemy>=2,<3" "psycopg[binary]>=3.2,<4" "httpx>=0.28,<1"

In [ ]:
import getpass
import json
import os
import random
import socket
import uuid
from pprint import pprint
from typing import Any

import httpx
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Connection, Engine, make_url
from sqlalchemy.exc import IntegrityError, OperationalError

RAILWAY_URL = "https://pixtallbackend-production-9ec3.up.railway.app/generate_image"
BACKEND_TIMEOUT_SECONDS = 180.0
MAX_ATTEMPTS = 3
LEASE_SECONDS = 300
WORKER_ID = f"colab-{socket.gethostname()}-{uuid.uuid4().hex[:8]}"

def read_colab_secret(name: str) -> str:
    """Read a secret without placing it in notebook source or output."""
    value = os.getenv(name)
    if value:
        return value
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except (ImportError, KeyError, Exception):
        value = None
    return value or getpass.getpass(f"Enter {name} (input is hidden): ")

def normalize_database_url(raw_url: str) -> str:
    """Select psycopg 3 and require TLS for the Supabase connection."""
    url = raw_url.strip()
    if url.startswith("postgres://"):
        url = url.replace("postgres://", "postgresql+psycopg://", 1)
    elif url.startswith("postgresql://"):
        url = url.replace("postgresql://", "postgresql+psycopg://", 1)
    elif not url.startswith("postgresql+psycopg://"):
        raise ValueError("DATABASE_URL must be a PostgreSQL connection string")
    if "sslmode=" not in url:
        url += ("&" if "?" in url else "?") + "sslmode=require"
    return url

DATABASE_URL = normalize_database_url(read_colab_secret("DATABASE_URL"))
DATABASE_HOST = make_url(DATABASE_URL).host or ""
engine = create_engine(DATABASE_URL, pool_pre_ping=True)

print({
    "database_url": "hidden Supabase PostgreSQL URL",
    "connection_mode": (
        "session_pooler" if ".pooler.supabase.com" in DATABASE_HOST else "direct"
    ),
    "railway_url": RAILWAY_URL,
    "worker_id": WORKER_ID,
    "lease_seconds": LEASE_SECONDS,
})

## 1. `check_database`

- **Purpose:** Confirm that Colab can reach the intended PostgreSQL database and that the lifecycle tables exist.
- **Input:** `db`, the configured SQLAlchemy `Engine`.
- **Output:** Database name, connected user, server time, and four table-existence flags.
- **Database effect:** Read-only.
- **Why it exists:** Stop the walkthrough early with a clear configuration or migration error.

In [ ]:
def check_database(db: Engine) -> dict[str, Any]:
    query = text("""
        SELECT
            current_database() AS database_name,
            current_user AS database_user,
            NOW() AS server_time,
            to_regclass('public.generation_jobs') IS NOT NULL AS has_generation_jobs,
            to_regclass('public.wallets') IS NOT NULL AS has_wallets,
            to_regclass('public.generation_outputs') IS NOT NULL AS has_generation_outputs,
            to_regclass('public.credit_transactions') IS NOT NULL AS has_credit_transactions
    """)
    with db.connect() as connection:
        return dict(connection.execute(query).mappings().one())

CHECK_INPUT = {"engine": "Supabase SQLAlchemy engine"}
try:
    CHECK_OUTPUT = check_database(engine)
except OperationalError as exc:
    is_direct_supabase = (
        DATABASE_HOST.startswith("db.")
        and DATABASE_HOST.endswith(".supabase.co")
    )
    if is_direct_supabase and "Network is unreachable" in str(exc):
        message = (
            "Colab cannot reach the Supabase direct IPv6 endpoint. In Supabase, "
            "open Connect > Session pooler, copy that PostgreSQL URL into the "
            "DATABASE_URL Colab secret, and rerun from the configuration cell."
        )
        raise RuntimeError(message) from None
    raise
print("INPUT")
pprint(CHECK_INPUT)
print("\nOUTPUT")
pprint(CHECK_OUTPUT)

required_flags = [
    key
    for key, value in CHECK_OUTPUT.items()
    if key.startswith("has_") and not value
]
if required_flags:
    message = f"Missing existing tables: {required_flags}. Run the service migration first."
    raise RuntimeError(message)

# Part A — API submission

The following helper functions keep notebook output readable without changing application data:

- `summarize_value(key, value)` replaces Base64 or very long strings with their character count.
- `safe_payload(payload)` applies that summary to every payload field.
- `safe_job_view(job)` copies a job and sanitizes only its nested request payload.

All three functions are pure: they return display-safe values and never modify Supabase or the original input.

In [ ]:
def summarize_value(key: str, value: Any) -> Any:
    if isinstance(value, str) and ("base64" in key.lower() or len(value) > 500):
        return f"<{len(value):,} characters hidden>"
    return value

def safe_payload(payload: dict[str, Any]) -> dict[str, Any]:
    return {key: summarize_value(key, value) for key, value in payload.items()}

def safe_job_view(job: dict[str, Any] | None) -> dict[str, Any] | None:
    if job is None:
        return None
    visible = dict(job)
    visible["request_payload"] = safe_payload(visible.get("request_payload") or {})
    return visible

## 2. `load_latest_submitted_job`

- **Purpose:** Locate the newest job that can still be processed after the frontend calls `POST /v1/generation-jobs`.
- **Input:** `db`, the SQLAlchemy engine.
- **Output:** A job dictionary containing identity, idempotency, pricing, state, request payload, and timestamps; otherwise `None`.
- **Database effect:** Read-only.
- **Why it exists:** Provide the real API-created job to every later cell instead of using sample data.

In [ ]:
def load_latest_submitted_job(db: Engine) -> dict[str, Any] | None:
    query = text("""
        SELECT
            id::text AS id, user_id::text AS user_id, idempotency_key,
            status::text AS status, quality::text AS quality, image_count,
            credit_cost, reserved_credits, consumed_credits, released_credits,
            attempt_count, request_payload, available_at, created_at, updated_at
        FROM generation_jobs
        WHERE status IN (
            CAST('QUEUED' AS job_status),
            CAST('RETRYING' AS job_status),
            CAST('PROCESSING' AS job_status)
        )
        ORDER BY created_at DESC
        LIMIT 1
    """)
    with db.connect() as connection:
        row = connection.execute(query).mappings().first()
        return dict(row) if row else None

SUBMISSION_INPUT = {
    "method": "POST",
    "path": "/v1/generation-jobs",
    "source": "frontend Generate button",
}
SUBMITTED_JOB = load_latest_submitted_job(engine)
print("INPUT")
pprint(SUBMISSION_INPUT)
print("\nOUTPUT")
pprint(safe_job_view(SUBMITTED_JOB))

if SUBMITTED_JOB is None:
    message = "No submitted job found. Stop make worker and click Generate in the frontend."
    raise RuntimeError(message)

## 3. `validate_generation_request`

- **Purpose:** Enforce the same field names, sizes, allowed choices, and image-count limits as the API request schema.
- **Input:** `payload`, the request dictionary stored on the job.
- **Output:** A copied, validated dictionary.
- **Failure:** Raises `ValueError` for missing, extra, malformed, or unsupported values.
- **Database effect:** None; this is a pure validation function.

In [ ]:
EXPECTED_REQUEST_FIELDS = {
    "product_image_base64", "model_image_base64", "product_category",
    "product_subcategory", "scene", "size", "model",
    "intended_use", "image_count", "quality",
}

def validate_generation_request(payload: dict[str, Any]) -> dict[str, Any]:
    missing = EXPECTED_REQUEST_FIELDS - payload.keys()
    extra = payload.keys() - EXPECTED_REQUEST_FIELDS
    if missing or extra:
        raise ValueError({"missing_fields": sorted(missing), "extra_fields": sorted(extra)})

    product_image = payload["product_image_base64"]
    model_image = payload["model_image_base64"]
    if not isinstance(product_image, str) or not 1 <= len(product_image) <= 15_000_000:
        raise ValueError("product_image_base64 must contain 1 to 15,000,000 characters")
    if model_image is not None and (
        not isinstance(model_image, str) or len(model_image) > 15_000_000
    ):
        raise ValueError("model_image_base64 must be null or at most 15,000,000 characters")

    for field in ("product_category", "product_subcategory", "scene"):
        value = payload[field]
        if not isinstance(value, str) or not 1 <= len(value) <= 80:
            raise ValueError(f"{field} must contain 1 to 80 characters")

    allowed_values = {
        "size": {"9:16", "3:4", "1:1", "4:5", "16:9"},
        "model": {"male", "female", "none"},
        "intended_use": {"marketplace", "website"},
        "quality": {"standard", "premium"},
    }
    for field, choices in allowed_values.items():
        if payload[field] not in choices:
            raise ValueError(f"{field} must be one of {sorted(choices)}")
    if not isinstance(payload["image_count"], int) or not 1 <= payload["image_count"] <= 4:
        raise ValueError("image_count must be an integer from 1 to 4")
    return dict(payload)

VALIDATION_INPUT = SUBMITTED_JOB["request_payload"]
VALIDATED_REQUEST = validate_generation_request(VALIDATION_INPUT)
print("INPUT")
pprint(safe_payload(VALIDATION_INPUT))
print("\nOUTPUT")
pprint({"valid": True, "normalized_request": safe_payload(VALIDATED_REQUEST)})

## 4. `load_authenticated_user`

- **Purpose:** Load the user selected by the API authentication boundary.
- **Inputs:** `db` and `authenticated_user_id`, taken from the submitted job rather than its request body.
- **Output:** The matching user ID, email, and name.
- **Failure:** Raises `RuntimeError` when the authenticated user does not exist.
- **Database effect:** Read-only. Email is account data, not trusted request identity.

In [ ]:
def load_authenticated_user(db: Engine, authenticated_user_id: str) -> dict[str, Any]:
    query = text("""
        SELECT id::text AS user_id, email, name
        FROM users WHERE id = :user_id
    """)
    with db.connect() as connection:
        row = connection.execute(query, {"user_id": authenticated_user_id}).mappings().first()
    if row is None:
        raise RuntimeError(f"Authenticated user {authenticated_user_id} does not exist")
    return dict(row)

IDENTITY_INPUT = {
    "auth_mode": "development",
    "user_id_selected_by_fastapi": SUBMITTED_JOB["user_id"],
    "email_from_request": None,
}
AUTHENTICATED_USER = load_authenticated_user(engine, SUBMITTED_JOB["user_id"])
print("INPUT")
pprint(IDENTITY_INPUT)
print("\nOUTPUT")
pprint(AUTHENTICATED_USER)

## 5. `load_active_plan` and `calculate_required_credits`

### `load_active_plan(db, user_id)`

Returns the user's newest active plan and its standard/premium per-image prices. It performs a read-only join across `subscriptions` and `plans` and fails when no active plan exists.

### `calculate_required_credits(plan, quality, image_count)`

Selects the plan price for the requested quality and returns the per-image price, image count, and total required credits. It is pure and performs no database operation.

In [ ]:
def load_active_plan(db: Engine, user_id: str) -> dict[str, Any]:
    query = text("""
        SELECT
            plans.id::text AS plan_id, plans.code, plans.name,
            plans.standard_image_credits, plans.premium_image_credits
        FROM plans
        JOIN subscriptions ON subscriptions.plan_id = plans.id
        WHERE subscriptions.user_id = :user_id
          AND subscriptions.status = 'active'
          AND plans.active IS TRUE
        ORDER BY subscriptions.starts_at DESC
        LIMIT 1
    """)
    with db.connect() as connection:
        row = connection.execute(query, {"user_id": user_id}).mappings().first()
    if row is None:
        raise RuntimeError("An active plan is required")
    return dict(row)

def calculate_required_credits(
    plan: dict[str, Any], quality: str, image_count: int
) -> dict[str, int]:
    price_field = (
        "standard_image_credits" if quality == "standard"
        else "premium_image_credits"
    )
    per_image_credits = int(plan[price_field])
    return {
        "per_image_credits": per_image_credits,
        "image_count": image_count,
        "required_credits": per_image_credits * image_count,
    }

PLAN_INPUT = {"user_id": AUTHENTICATED_USER["user_id"]}
ACTIVE_PLAN = load_active_plan(engine, AUTHENTICATED_USER["user_id"])
PRICING_INPUT = {
    "plan": ACTIVE_PLAN,
    "quality": VALIDATED_REQUEST["quality"],
    "image_count": VALIDATED_REQUEST["image_count"],
}
PRICING_OUTPUT = calculate_required_credits(
    ACTIVE_PLAN, VALIDATED_REQUEST["quality"], VALIDATED_REQUEST["image_count"]
)
print("LOAD ACTIVE PLAN - INPUT")
pprint(PLAN_INPUT)
print("LOAD ACTIVE PLAN - OUTPUT")
pprint(ACTIVE_PLAN)
print("\nCALCULATE CREDITS - INPUT")
pprint(PRICING_INPUT)
print("CALCULATE CREDITS - OUTPUT")
pprint(PRICING_OUTPUT)

if PRICING_OUTPUT["required_credits"] != SUBMITTED_JOB["credit_cost"]:
    raise RuntimeError("Calculated price does not match the persisted job price")

## 6. `find_job_by_idempotency_key` and `submit_job`

### `find_job_by_idempotency_key(connection, user_id, idempotency_key)`

Returns the existing job for one user/key pair or `None`. This read is the first idempotency check and is also used after a concurrent unique-key conflict.

### `submit_job(db, user_id, idempotency_key, request)`

Runs one atomic transaction: return an idempotent match, load the active plan, calculate cost, lock the wallet, check available credits, insert a `QUEUED` job, update wallet balances, and insert the `RESERVED` ledger entry. Its output is the existing or newly created job plus `created_by_this_call`.

Because FastAPI already submitted this frontend request, the notebook call should return `created_by_this_call = False`. This proves idempotency and prevents a second reservation.

In [ ]:
def find_job_by_idempotency_key(
    connection: Connection, user_id: str, idempotency_key: str
) -> dict[str, Any] | None:
    query = text("""
        SELECT
            id::text AS id, user_id::text AS user_id, idempotency_key,
            quality::text AS quality, image_count, credit_cost, reserved_credits,
            consumed_credits, released_credits, status::text AS status,
            request_payload, attempt_count, safe_error, created_at, updated_at
        FROM generation_jobs
        WHERE user_id = :user_id AND idempotency_key = :idempotency_key
    """)
    row = connection.execute(query, {
        "user_id": user_id,
        "idempotency_key": idempotency_key,
    }).mappings().first()
    return dict(row) if row else None

def submit_job(
    db: Engine,
    *,
    user_id: str,
    idempotency_key: str,
    request: dict[str, Any],
) -> dict[str, Any]:
    try:
        with db.begin() as connection:
            existing = find_job_by_idempotency_key(
                connection, user_id, idempotency_key
            )
            if existing is not None:
                existing["created_by_this_call"] = False
                existing["reason"] = "Idempotency key already exists"
                return existing

            plan = connection.execute(text("""
                SELECT standard_image_credits, premium_image_credits
                FROM plans
                JOIN subscriptions ON subscriptions.plan_id = plans.id
                WHERE subscriptions.user_id = :user_id
                  AND subscriptions.status = 'active'
                  AND plans.active IS TRUE
                ORDER BY subscriptions.starts_at DESC
                LIMIT 1
            """), {"user_id": user_id}).mappings().first()
            if plan is None:
                raise RuntimeError("An active plan is required")

            quality = request["quality"]
            image_count = int(request["image_count"])
            if quality == "standard":
                per_image_credits = int(plan["standard_image_credits"])
            elif quality == "premium":
                per_image_credits = int(plan["premium_image_credits"])
            else:
                raise ValueError(f"Unsupported quality: {quality}")
            required_credits = per_image_credits * image_count

            wallet = connection.execute(text("""
                SELECT available_credits, reserved_credits
                FROM wallets
                WHERE user_id = :user_id
                FOR UPDATE
            """), {"user_id": user_id}).mappings().first()
            if wallet is None:
                raise RuntimeError("Wallet not found")

            available_before = int(wallet["available_credits"])
            reserved_before = int(wallet["reserved_credits"])
            if available_before < required_credits:
                message = (
                    f"Generation requires {required_credits} credits, but only "
                    f"{available_before} are available"
                )
                raise RuntimeError(message)

            job_id = str(uuid.uuid4())
            available_after = available_before - required_credits
            reserved_after = reserved_before + required_credits

            connection.execute(text("""
                INSERT INTO generation_jobs (
                    id, user_id, idempotency_key, quality, image_count,
                    credit_cost, reserved_credits, status, request_payload
                )
                VALUES (
                    :job_id, :user_id, :idempotency_key,
                    CAST(:quality AS image_quality), :image_count,
                    :credit_cost, :reserved_credits,
                    CAST('QUEUED' AS job_status),
                    CAST(:request_payload AS JSONB)
                )
            """), {
                "job_id": job_id,
                "user_id": user_id,
                "idempotency_key": idempotency_key,
                "quality": quality.upper(),
                "image_count": image_count,
                "credit_cost": required_credits,
                "reserved_credits": required_credits,
                "request_payload": json.dumps(request),
            })

            connection.execute(text("""
                UPDATE wallets
                SET available_credits = :available_after,
                    reserved_credits = :reserved_after,
                    updated_at = NOW()
                WHERE user_id = :user_id
            """), {
                "available_after": available_after,
                "reserved_after": reserved_after,
                "user_id": user_id,
            })

            connection.execute(text("""
                INSERT INTO credit_transactions (
                    id, user_id, job_id, type, credits,
                    available_balance_after, reserved_balance_after
                )
                VALUES (
                    :transaction_id, :user_id, :job_id,
                    CAST('RESERVED' AS transaction_type), :credits,
                    :available_after, :reserved_after
                )
            """), {
                "transaction_id": str(uuid.uuid4()),
                "user_id": user_id,
                "job_id": job_id,
                "credits": required_credits,
                "available_after": available_after,
                "reserved_after": reserved_after,
            })

            created = find_job_by_idempotency_key(
                connection, user_id, idempotency_key
            )
            if created is None:
                raise RuntimeError("Job creation did not complete")
            created["created_by_this_call"] = True
            created["wallet_before"] = {
                "available_credits": available_before,
                "reserved_credits": reserved_before,
            }
            created["wallet_after"] = {
                "available_credits": available_after,
                "reserved_credits": reserved_after,
            }
            return created
    except IntegrityError:
        with db.connect() as connection:
            duplicate = find_job_by_idempotency_key(
                connection, user_id, idempotency_key
            )
        if duplicate is None:
            raise
        duplicate["created_by_this_call"] = False
        duplicate["reason"] = "Concurrent request already created this job"
        return duplicate

In [ ]:
SUBMIT_JOB_INPUT = {
    "user_id": SUBMITTED_JOB["user_id"],
    "idempotency_key": SUBMITTED_JOB["idempotency_key"],
    "request": VALIDATED_REQUEST,
}
SUBMIT_JOB_OUTPUT = submit_job(
    engine,
    user_id=SUBMIT_JOB_INPUT["user_id"],
    idempotency_key=SUBMIT_JOB_INPUT["idempotency_key"],
    request=SUBMIT_JOB_INPUT["request"],
)
print("INPUT")
pprint({
    **SUBMIT_JOB_INPUT,
    "request": safe_payload(SUBMIT_JOB_INPUT["request"]),
})
print("\nOUTPUT")
pprint(safe_job_view(SUBMIT_JOB_OUTPUT))

## 7. `inspect_atomic_reservation`

- **Purpose:** Explain the committed credit movement using persisted evidence.
- **Inputs:** `db` and `job_id`.
- **Output:** Reconstructed wallet balances before reservation, the debit/reserve operation, balances afterward, and the underlying job/ledger/wallet fields.
- **Failure:** Raises `RuntimeError` if the job has no `RESERVED` transaction.
- **Database effect:** Read-only; it never performs the reservation again.

In [ ]:
def inspect_atomic_reservation(db: Engine, job_id: str) -> dict[str, Any]:
    query = text("""
        SELECT
            jobs.id::text AS job_id, jobs.status::text AS job_status,
            jobs.credit_cost, jobs.reserved_credits AS job_reserved_credits,
            transactions.type::text AS transaction_type,
            transactions.credits AS ledger_credits,
            transactions.available_balance_after,
            transactions.reserved_balance_after,
            wallets.available_credits AS current_available_credits,
            wallets.reserved_credits AS current_reserved_credits
        FROM generation_jobs AS jobs
        JOIN wallets ON wallets.user_id = jobs.user_id
        JOIN credit_transactions AS transactions ON transactions.job_id = jobs.id
        WHERE jobs.id = :job_id
          AND transactions.type = CAST('RESERVED' AS transaction_type)
    """)
    with db.connect() as connection:
        row = connection.execute(query, {"job_id": job_id}).mappings().first()
    if row is None:
        raise RuntimeError("The job has no RESERVED credit transaction")
    reservation = dict(row)
    credits = int(reservation["ledger_credits"])
    available_after = int(reservation["available_balance_after"])
    reserved_after = int(reservation["reserved_balance_after"])
    return {
        "wallet_before": {
            "available_credits": available_after + credits,
            "reserved_credits": reserved_after - credits,
        },
        "operation": {
            "available_credits_change": -credits,
            "reserved_credits_change": credits,
        },
        "wallet_after_reservation": {
            "available_credits": available_after,
            "reserved_credits": reserved_after,
        },
        "persisted_evidence": reservation,
    }

RESERVATION_INPUT = {
    "job_id": SUBMITTED_JOB["id"],
    "required_credits": PRICING_OUTPUT["required_credits"],
}
RESERVATION_OUTPUT = inspect_atomic_reservation(engine, SUBMITTED_JOB["id"])
print("INPUT")
pprint(RESERVATION_INPUT)
print("\nOUTPUT")
pprint(RESERVATION_OUTPUT)

## 8. `build_accepted_response`

- **Purpose:** Represent the response returned immediately after durable job submission.
- **Input:** The queued job dictionary.
- **Output:** HTTP status `202`, the job-specific `Location` header, and the public response body.
- **Database effect:** None; this is a pure response-mapping function.
- **Why it exists:** Let the frontend begin polling without holding the generation request open.

In [ ]:
def build_accepted_response(job: dict[str, Any]) -> dict[str, Any]:
    body = {
        "id": job["id"],
        "status": job["status"].lower(),
        "quality": job["quality"].lower(),
        "image_count": job["image_count"],
        "credit_cost": job["credit_cost"],
        "reserved_credits": job["reserved_credits"],
        "consumed_credits": job["consumed_credits"],
        "released_credits": job["released_credits"],
        "attempt_count": job["attempt_count"],
        "safe_error": None,
        "created_at": job["created_at"],
        "updated_at": job["updated_at"],
        "outputs": [],
    }
    return {
        "http_status": 202,
        "headers": {"Location": f"/v1/generation-jobs/{job['id']}"},
        "body": body,
    }

ACCEPTED_RESPONSE_INPUT = {"queued_job": safe_job_view(SUBMITTED_JOB)}
ACCEPTED_RESPONSE_OUTPUT = build_accepted_response(SUBMITTED_JOB)
print("INPUT")
pprint(ACCEPTED_RESPONSE_INPUT)
print("\nOUTPUT")
pprint(ACCEPTED_RESPONSE_OUTPUT)

# Part B — Worker execution

The API has completed its responsibility once it returns a durable job ID. The remaining functions perform the worker responsibility: claim one job, invoke the backend, retry when appropriate, and settle credits exactly once.

## 9. `peek_next_job`

- **Purpose:** Preview the oldest job currently eligible for worker processing.
- **Input:** `db`, the SQLAlchemy engine.
- **Output:** A queued/retrying job, or a processing job whose lease expired; otherwise `None`.
- **Database effect:** Read-only and does not reserve the job.
- **Why it exists:** Make the claim candidate visible before the atomic state change.

In [ ]:
def peek_next_job(db: Engine) -> dict[str, Any] | None:
    query = text("""
        SELECT
            id::text AS id, user_id::text AS user_id, status::text AS status,
            quality::text AS quality, image_count, credit_cost, reserved_credits,
            attempt_count, available_at, lease_owner, lease_expires_at,
            request_payload, created_at
        FROM generation_jobs
        WHERE available_at <= NOW()
          AND (
              status IN (CAST('QUEUED' AS job_status), CAST('RETRYING' AS job_status))
              OR (status = CAST('PROCESSING' AS job_status) AND lease_expires_at < NOW())
          )
        ORDER BY created_at
        LIMIT 1
    """)
    with db.connect() as connection:
        row = connection.execute(query).mappings().first()
        return dict(row) if row else None

In [ ]:
PEEK_INPUT = {"database": "Supabase", "operation": "read only"}
PEEKED_JOB = peek_next_job(engine)
print("INPUT")
pprint(PEEK_INPUT)
print("\nOUTPUT")
pprint(safe_job_view(PEEKED_JOB))

if PEEKED_JOB is None:
    message = "No job found. Click Generate in the frontend and stop make worker."
    raise RuntimeError(message)

## 10. `claim_job`

- **Inputs:** `db`, a unique `worker_id`, and `lease_seconds`.
- **Output:** The claimed job or `None` when no eligible row remains.
- **Database effect:** Atomically selects with `FOR UPDATE SKIP LOCKED`, changes status to `PROCESSING`, increments attempts, and records lease ownership/expiry.
- **Concurrency rule:** Competing workers skip the locked row instead of processing the same job.
- **Recovery rule:** An expired lease makes a crashed worker's job claimable again.

In [ ]:
def claim_job(db: Engine, worker_id: str, lease_seconds: int) -> dict[str, Any] | None:
    query = text("""
        WITH candidate AS (
            SELECT id
            FROM generation_jobs
            WHERE available_at <= NOW()
              AND (
                  status IN (CAST('QUEUED' AS job_status), CAST('RETRYING' AS job_status))
                  OR (status = CAST('PROCESSING' AS job_status) AND lease_expires_at < NOW())
              )
            ORDER BY created_at
            FOR UPDATE SKIP LOCKED
            LIMIT 1
        )
        UPDATE generation_jobs AS job
        SET status = CAST('PROCESSING' AS job_status),
            attempt_count = job.attempt_count + 1,
            lease_owner = :worker_id,
            lease_expires_at = NOW() + make_interval(secs => :lease_seconds),
            updated_at = NOW()
        FROM candidate
        WHERE job.id = candidate.id
        RETURNING
            job.id::text AS id, job.user_id::text AS user_id, job.status::text AS status,
            job.quality::text AS quality, job.image_count, job.credit_cost,
            job.reserved_credits, job.attempt_count, job.request_payload,
            job.lease_owner, job.lease_expires_at, job.created_at
    """)
    with db.begin() as connection:
        row = connection.execute(
            query, {"worker_id": worker_id, "lease_seconds": float(lease_seconds)}
        ).mappings().first()
        return dict(row) if row else None

In [ ]:
CLAIM_INPUT = {"worker_id": WORKER_ID, "lease_seconds": LEASE_SECONDS}
CLAIMED_JOB = claim_job(engine, WORKER_ID, LEASE_SECONDS)
print("INPUT")
pprint(CLAIM_INPUT)
print("\nOUTPUT")
pprint(safe_job_view(CLAIMED_JOB))

if CLAIMED_JOB is None:
    message = (
        "The job was claimed elsewhere. Stop make worker, create another frontend "
        "job, and rerun from the peek cell."
    )
    raise RuntimeError(message)

## 11. `load_user_email` and `build_railway_payload`

### `load_user_email(db, user_id)`

Reads the account email for the job owner. It returns a string or `None` and never accepts email as request identity.

### `build_railway_payload(job, user_email)`

Maps the stored snake_case request into the existing Railway endpoint's field names and adds the trusted account email. It returns a new payload dictionary and performs no I/O.

In [ ]:
def load_user_email(db: Engine, user_id: str) -> str | None:
    with db.connect() as connection:
        return connection.execute(
            text("SELECT email FROM users WHERE id = :user_id"),
            {"user_id": user_id},
        ).scalar_one_or_none()

def build_railway_payload(job: dict[str, Any], user_email: str | None) -> dict[str, Any]:
    request = job["request_payload"]
    return {
        "productImageBase64": request["product_image_base64"],
        "modelImageBase64": request.get("model_image_base64"),
        "productCategory": request["product_category"],
        "productSubcategory": request["product_subcategory"],
        "scene": request["scene"],
        "size": request["size"],
        "model": request["model"],
        "intendUse": request["intended_use"],
        "numImages": int(job["image_count"]),
        "email": user_email,
    }

USER_EMAIL_INPUT = {"user_id": CLAIMED_JOB["user_id"]}
USER_EMAIL = load_user_email(engine, CLAIMED_JOB["user_id"])
RAILWAY_PAYLOAD = build_railway_payload(CLAIMED_JOB, USER_EMAIL)

print("LOAD USER EMAIL - INPUT")
pprint(USER_EMAIL_INPUT)
print("LOAD USER EMAIL - OUTPUT")
pprint({"email": USER_EMAIL})
print("\nBUILD RAILWAY PAYLOAD - INPUT")
pprint({"job": safe_job_view(CLAIMED_JOB), "user_email": USER_EMAIL})
print("BUILD RAILWAY PAYLOAD - OUTPUT")
pprint({key: summarize_value(key, value) for key, value in RAILWAY_PAYLOAD.items()})

## 12. `BackendCallError`, `call_railway_backend`, and `safe_backend_result`

- `BackendCallError(message, retryable)` carries a safe failure message and explicitly states whether another attempt is allowed.
- `call_railway_backend(url, payload, timeout_seconds)` performs the real streaming POST, classifies HTTP/network failures, parses each NDJSON line, and returns independent `images` and `errors` collections. It performs external network I/O but does not write Supabase.
- `safe_backend_result(result)` replaces generated image data with character counts for display. It is pure and does not alter the real results used by settlement.

In [ ]:
class BackendCallError(Exception):
    def __init__(self, message: str, *, retryable: bool) -> None:
        super().__init__(message)
        self.retryable = retryable

def call_railway_backend(
    url: str, payload: dict[str, Any], timeout_seconds: float
) -> dict[str, Any]:
    image_count = int(payload["numImages"])
    images: list[str | None] = [None] * image_count
    errors: dict[int, str] = {}
    try:
        timeout = httpx.Timeout(timeout_seconds)
        with (
            httpx.Client(timeout=timeout) as client,
            client.stream("POST", url, json=payload) as response,
        ):
                if response.status_code == 429 or response.status_code >= 500:
                    raise BackendCallError(
                        f"Railway backend temporarily unavailable ({response.status_code})",
                        retryable=True,
                    )
                if response.status_code >= 400:
                    raise BackendCallError(
                        f"Railway backend rejected the request ({response.status_code})",
                        retryable=False,
                    )
                next_index = 0
                for line in response.iter_lines():
                    if not line.strip():
                        continue
                    try:
                        item = json.loads(line)
                    except json.JSONDecodeError:
                        if next_index < image_count:
                            errors[next_index] = "Railway backend returned an unreadable result"
                            next_index += 1
                        continue
                    index = item.get("index", next_index)
                    if not isinstance(index, int) or not 0 <= index < image_count:
                        continue
                    image = item.get("image") or item.get("imageUrl") or item.get("imageBase64")
                    error = item.get("error") or item.get("detail")
                    if isinstance(image, str) and image:
                        images[index] = image
                    elif isinstance(error, str):
                        errors[index] = error[:500]
                    else:
                        errors[index] = "Railway backend returned no image"
                    next_index = max(next_index, index + 1)
    except httpx.RequestError as exc:
        raise BackendCallError("Could not reach the Railway backend", retryable=True) from exc

    for index, image in enumerate(images):
        if image is None and index not in errors:
            errors[index] = "Railway backend returned no result"
    return {"images": images, "errors": errors}

def safe_backend_result(result: dict[str, Any] | None) -> dict[str, Any] | None:
    if result is None:
        return None
    return {
        "images": [
            None if image is None else f"<generated result: {len(image):,} characters>"
            for image in result["images"]
        ],
        "errors": result["errors"],
    }

In [ ]:
BACKEND_INPUT = {
    "url": RAILWAY_URL,
    "payload": {key: summarize_value(key, value) for key, value in RAILWAY_PAYLOAD.items()},
    "timeout_seconds": BACKEND_TIMEOUT_SECONDS,
}
BACKEND_RESULT = None
BACKEND_FAILURE = None

print("INPUT")
pprint(BACKEND_INPUT)
try:
    BACKEND_RESULT = call_railway_backend(
        RAILWAY_URL, RAILWAY_PAYLOAD, BACKEND_TIMEOUT_SECONDS
    )
except BackendCallError as exc:
    BACKEND_FAILURE = {"message": str(exc), "retryable": exc.retryable}

print("\nOUTPUT")
pprint(safe_backend_result(BACKEND_RESULT) or BACKEND_FAILURE)

## 13. `schedule_retry` and `settle_job`

### `schedule_retry(db, job_id, delay_seconds, safe_error)`

Moves a processing job to `RETRYING`, sets its next availability time, stores a safe error, and clears the lease. It returns the updated scheduling fields.

### `settle_job(db, job_id, results, errors, terminal_error)`

Locks the job and wallet; ignores already-terminal jobs; stores one output row per requested image; consumes credits for successes; releases unused reserved credits; updates wallet/job balances; and inserts `USED`/`RELEASED` ledger rows in one transaction. It returns the final status and settlement totals.

The execution cell chooses retry only for retryable failures below `MAX_ATTEMPTS`; every other outcome is settled.

In [ ]:
TERMINAL_STATUSES = {"COMPLETED", "PARTIALLY_COMPLETED", "FAILED"}

def schedule_retry(
    db: Engine, job_id: str, delay_seconds: float, safe_error: str
) -> dict[str, Any]:
    query = text("""
        UPDATE generation_jobs
        SET status = CAST('RETRYING' AS job_status),
            available_at = NOW() + make_interval(secs => :delay_seconds),
            safe_error = :safe_error,
            lease_owner = NULL,
            lease_expires_at = NULL,
            updated_at = NOW()
        WHERE id = :job_id AND status = CAST('PROCESSING' AS job_status)
        RETURNING id::text AS id, status::text AS status, attempt_count, available_at, safe_error
    """)
    with db.begin() as connection:
        row = connection.execute(query, {
            "job_id": job_id,
            "delay_seconds": float(delay_seconds),
            "safe_error": safe_error[:500],
        }).mappings().first()
    return dict(row) if row else {"job_id": job_id, "changed": False}

def settle_job(
    db: Engine,
    job_id: str,
    results: list[str | None],
    errors: dict[int, str],
    terminal_error: str | None,
) -> dict[str, Any]:
    with db.begin() as connection:
        job_row = connection.execute(text("""
            SELECT id::text AS id, user_id::text AS user_id, status::text AS status,
                   image_count, credit_cost, reserved_credits
            FROM generation_jobs WHERE id = :job_id FOR UPDATE
        """), {"job_id": job_id}).mappings().first()
        if job_row is None:
            raise RuntimeError(f"Job {job_id} does not exist")
        job = dict(job_row)
        if job["status"] in TERMINAL_STATUSES:
            return {"job_id": job_id, "status": job["status"], "changed": False}

        wallet_row = connection.execute(text("""
            SELECT available_credits, reserved_credits, lifetime_consumed_credits
            FROM wallets WHERE user_id = :user_id FOR UPDATE
        """), {"user_id": job["user_id"]}).mappings().first()
        if wallet_row is None:
            raise RuntimeError(f"Wallet for user {job['user_id']} does not exist")
        wallet = dict(wallet_row)

        image_count = int(job["image_count"])
        successful = sum(
            index < len(results) and results[index] is not None
            for index in range(image_count)
        )
        per_image_cost = int(job["credit_cost"]) // image_count
        consumed = successful * per_image_cost
        released = int(job["reserved_credits"]) - consumed
        status = (
            "COMPLETED" if successful == image_count
            else "PARTIALLY_COMPLETED" if successful
            else "FAILED"
        )

        for index in range(image_count):
            result_reference = results[index] if index < len(results) else None
            connection.execute(text("""
                INSERT INTO generation_outputs
                    (id, job_id, output_index, result_reference, error)
                VALUES (:id, :job_id, :output_index, :result_reference, :error)
                ON CONFLICT (job_id, output_index) DO NOTHING
            """), {
                "id": str(uuid.uuid4()),
                "job_id": job_id,
                "output_index": index,
                "result_reference": result_reference,
                "error": errors.get(index),
            })

        available_after = int(wallet["available_credits"]) + released
        reserved_after = int(wallet["reserved_credits"]) - int(job["reserved_credits"])
        if reserved_after < 0:
            message = "Wallet reserved balance is inconsistent; transaction was rolled back"
            raise RuntimeError(message)

        connection.execute(text("""
            UPDATE wallets
            SET available_credits = :available_after,
                reserved_credits = :reserved_after,
                lifetime_consumed_credits = lifetime_consumed_credits + :consumed,
                updated_at = NOW()
            WHERE user_id = :user_id
        """), {
            "available_after": available_after,
            "reserved_after": reserved_after,
            "consumed": consumed,
            "user_id": job["user_id"],
        })

        connection.execute(text("""
            UPDATE generation_jobs
            SET status = CAST(:status AS job_status),
                consumed_credits = :consumed,
                released_credits = :released,
                safe_error = :safe_error,
                lease_owner = NULL,
                lease_expires_at = NULL,
                updated_at = NOW()
            WHERE id = :job_id
        """), {
            "status": status, "consumed": consumed, "released": released,
            "safe_error": terminal_error[:500] if terminal_error else None,
            "job_id": job_id,
        })

        for transaction_type, credits in (("USED", consumed), ("RELEASED", released)):
            if credits == 0:
                continue
            connection.execute(text("""
                INSERT INTO credit_transactions
                    (id, user_id, job_id, type, credits,
                     available_balance_after, reserved_balance_after)
                VALUES
                    (:id, :user_id, :job_id, CAST(:transaction_type AS transaction_type),
                     :credits, :available_after, :reserved_after)
                ON CONFLICT (job_id, type) DO NOTHING
            """), {
                "id": str(uuid.uuid4()), "user_id": job["user_id"],
                "job_id": job_id, "transaction_type": transaction_type,
                "credits": credits, "available_after": available_after,
                "reserved_after": reserved_after,
            })

        return {
            "job_id": job_id, "status": status, "successful_images": successful,
            "consumed_credits": consumed, "released_credits": released,
            "available_credits_after": available_after,
            "reserved_credits_after": reserved_after, "changed": True,
        }

In [ ]:
if BACKEND_FAILURE is not None:
    should_retry = BACKEND_FAILURE["retryable"] and CLAIMED_JOB["attempt_count"] < MAX_ATTEMPTS
    if should_retry:
        delay = min(60.0, 2 ** (CLAIMED_JOB["attempt_count"] - 1)) + random.uniform(0, 1)
        FINAL_INPUT = {
            "function": "schedule_retry", "job_id": CLAIMED_JOB["id"],
            "delay_seconds": round(delay, 2), "safe_error": BACKEND_FAILURE["message"],
        }
        FINAL_OUTPUT = schedule_retry(
            engine, CLAIMED_JOB["id"], delay, BACKEND_FAILURE["message"]
        )
    else:
        failure_errors = {
            index: BACKEND_FAILURE["message"]
            for index in range(CLAIMED_JOB["image_count"])
        }
        FINAL_INPUT = {
            "function": "settle_job", "job_id": CLAIMED_JOB["id"],
            "results": [None] * CLAIMED_JOB["image_count"],
            "errors": failure_errors, "terminal_error": BACKEND_FAILURE["message"],
        }
        FINAL_OUTPUT = settle_job(
            engine, CLAIMED_JOB["id"], [None] * CLAIMED_JOB["image_count"],
            failure_errors, BACKEND_FAILURE["message"],
        )
else:
    terminal_error = "Some images could not be generated" if BACKEND_RESULT["errors"] else None
    FINAL_INPUT = {
        "function": "settle_job", "job_id": CLAIMED_JOB["id"],
        "results": safe_backend_result(BACKEND_RESULT)["images"],
        "errors": BACKEND_RESULT["errors"], "terminal_error": terminal_error,
    }
    FINAL_OUTPUT = settle_job(
        engine, CLAIMED_JOB["id"], BACKEND_RESULT["images"],
        BACKEND_RESULT["errors"], terminal_error,
    )

print("INPUT")
pprint(FINAL_INPUT)
print("\nOUTPUT")
pprint(FINAL_OUTPUT)

## 14. `inspect_final_state`

- **Purpose:** Verify the result visible to the frontend after retry or settlement.
- **Inputs:** `db` and `job_id`.
- **Output:** The job, owner's wallet, ordered generation outputs, and job-specific credit transactions.
- **Database effect:** Read-only. Result references are sanitized only in the returned display structure.

In [ ]:
def inspect_final_state(db: Engine, job_id: str) -> dict[str, Any]:
    with db.connect() as connection:
        job = connection.execute(text("""
            SELECT id::text AS id, user_id::text AS user_id, status::text AS status,
                   image_count, reserved_credits, consumed_credits, released_credits,
                   attempt_count, safe_error, available_at, updated_at
            FROM generation_jobs WHERE id = :job_id
        """), {"job_id": job_id}).mappings().one()
        wallet = connection.execute(text("""
            SELECT available_credits, reserved_credits, lifetime_consumed_credits, updated_at
            FROM wallets WHERE user_id = :user_id
        """), {"user_id": str(job["user_id"])}).mappings().one()
        outputs = connection.execute(text("""
            SELECT output_index, result_reference, error, created_at
            FROM generation_outputs WHERE job_id = :job_id ORDER BY output_index
        """), {"job_id": job_id}).mappings().all()
        ledger = connection.execute(text("""
            SELECT type::text AS type, credits, available_balance_after,
                   reserved_balance_after, created_at
            FROM credit_transactions WHERE job_id = :job_id ORDER BY created_at
        """), {"job_id": job_id}).mappings().all()
    safe_outputs = []
    for output in outputs:
        item = dict(output)
        item["result_reference"] = summarize_value("result_reference", item["result_reference"])
        safe_outputs.append(item)
    return {
        "job": dict(job), "wallet": dict(wallet),
        "outputs": safe_outputs, "credit_transactions": [dict(row) for row in ledger],
    }

INSPECT_INPUT = {"job_id": CLAIMED_JOB["id"]}
INSPECT_OUTPUT = inspect_final_state(engine, CLAIMED_JOB["id"])
print("INPUT")
pprint(INSPECT_INPUT)
print("\nOUTPUT")
pprint(INSPECT_OUTPUT)

## Interpreting the final output

- `COMPLETED`: every requested output succeeded and all reserved credits were consumed.
- `PARTIALLY_COMPLETED`: at least one output succeeded; only successful-image credits were consumed and the remainder was released.
- `FAILED`: no output succeeded and unused reserved credits were released.
- `RETRYING`: wait until `available_at`, then rerun from `peek_next_job`.

## Execution boundary

The claim, retry, and settlement functions write to the real database. Keep `make worker` stopped during the walkthrough, keep `DATABASE_URL` in Colab Secrets, and use the automated worker—not this notebook—for normal continuous processing.